# Notebook to do all the API requests

## Import packages

In [1]:
import os

import numpy as np
import pandas as pd

import requests
import time

## Load the API keys from the .env file

In [2]:
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

## Set API final variables

In [3]:
TOKEN = "Bearer " + os.getenv("TMDB_TOKEN")
API_URL = "https://api.themoviedb.org/3/"

## Function to get the JSON from the request

In [5]:
HEADERS= {
    "accept": "application/json",
    "Authorization": TOKEN
}

def make_request(url):
    try:
        response = requests.get(url, headers=HEADERS)
    except requests.RequestException as error:
        raise RuntimeError(f"API request failed: {type(error).__name__}")
    
    if not response.ok:
        raise RuntimeError(f"API request failed with HTTP status {response.status_code}")

    return response.json()

## Create directories /dat/raw

In [6]:
os.makedirs("data/raw", exist_ok=True)

## Get all the movies and save info in a CSV file

In [4]:
# Set first request final variables
NUM_PAGES = 10 # Number of pages to requests movies
VOTE_COUNT = 10000 # Minimum number of votes to get the movie
SORT_BY = "popularity.desc" # Get movies sorted by popularity
URL_GET_MOVIES = API_URL + f"discover/movie?include_adult=false&include_video=false&language=en-US&sort_by={SORT_BY}&vote_count.gte={VOTE_COUNT}"
DROP_COLUMNS = ["adult", "backdrop_path", "original_language", "original_title", "overview", "poster_path", "popularity", "softcore", "video", "genre_ids"]

In [7]:
first_time = True # Variable to add the header to the file only the first time
path_csv = "data/raw/movies.csv"

for page in range(1, NUM_PAGES+1):
    url = URL_GET_MOVIES + f"&page={page}"
    movies = pd.DataFrame(make_request(url)["results"])
    movies.drop(DROP_COLUMNS, axis=1, inplace=True)
    
    movies.to_csv(path_csv, sep = ",", header = True if first_time else False, index = False, mode = "a", na_rep = "N/A")
    first_time = False

## Get extra info for each movie and save it in a CSV file

In [68]:
# Read CSV in a DataFrame
path_csv = "data/raw/movies.csv"
df_movies = pd.read_csv(path_csv)
display(df_movies.head())

,id,title,release_date,vote_average,vote_count
0,9479,The Nightmare Before Christmas,1993-10-09,7.852,10786
1,299536,Avengers: Infinity War,2018-04-25,8.246,33228
2,299534,Avengers: Endgame,2019-04-24,8.247,29087
3,157336,Interstellar,2014-11-04,8.500,41598
4,354912,Coco,2017-10-27,8.201,21602


In [69]:
PRUEBA = False

In [70]:
path_csv = "data/raw/extra_info.csv"
first_time = True # Variable to add the header to the file only the first time
num_to_save = 20 # Same number as movies per page in first request

columns = ["id", "budget", "revenue", "origin_country", "runtime", "genres", "directors", "cast"] # info to save for each movie
df_extra_info = pd.DataFrame(columns=columns)

if PRUEBA:
    range_ = range(1)
else:
    range_ = range(len(df_movies.index))

for i in range_:
    # Save to file every num_to_save movies and clean dataframe
    if (i>0 and i%num_to_save == 0):
        df_extra_info.to_csv(path_csv, sep = ",", header = True if first_time else False, index = False, mode = "a", na_rep = "N/A")
        first_time = False
        df_extra_info = pd.DataFrame(columns=columns)
    
    id_movie = df_movies["id"].iloc[i]
    url = API_URL + f"movie/{id_movie}?language=en-US&append_to_response=credits"
    answer = make_request(url)

    # Only save 'columns' info
    # Only save first origin country
    info = [answer[column] if column != "origin_country" else answer[column][0] for column in columns if (column != "genres" and column != "directors" and column != "cast")]

    # Save all genres
    genres = [genre["name"] for genre in answer["genres"]]
    info.append(genres)

    # Only save first two directors info
    directors = [[director["id"], director["name"], director["original_name"], director["gender"]] for director in [crew for crew in answer["credits"]["crew"] if crew["job"] == "Director"][:2]]
    info.append(directors)

    # Only save first five cast info
    cast = [[cast["id"], cast["gender"], cast["name"], cast["original_name"], cast["character"]] for cast in answer["credits"]["cast"][:5]]
    info.append(cast)

    aux = pd.DataFrame(info).T
    aux.columns = columns

    df_extra_info = pd.concat([df_extra_info, aux], ignore_index=True)

# Save dataframe in csv one last time
df_extra_info.to_csv(path_csv, sep = ",", header = True if first_time else False, index = False, mode = "a", na_rep = "N/A")

## Get all genres in API and save it in a CSV file

In [67]:
url = API_URL + f"/genre/movie/list"
answer = make_request(url)

df_genres = pd.DataFrame(answer["genres"])

path_csv = "data/genres.csv"
df_genres.to_csv(path_csv, sep = ",", header = True, index = False, mode = "a", na_rep = "N/A")